## 1. Setup and Path Configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
import numpy as np
from IPython.display import Image, display
from ultralytics import YOLO
import random

# ================= CONFIGURAÇÃO DE CAMINHOS =================
BASE_DIR = Path("/workspace") 

LOG_DIR = BASE_DIR / "logs/ph2_finetuning"
TEST_IMAGES_DIR = BASE_DIR / "datasets/ph2/test/images"
WEIGHTS_PATH = LOG_DIR / "weights/best.pt"
RESULTS_CSV = LOG_DIR / "results.csv"

## 2. Data Loading

# O YOLO compila todas as métricas de treino e validação num único CSV
if RESULTS_CSV.exists():
    df_results = pd.read_csv(RESULTS_CSV)
    
    # Remove os espaços em branco que o YOLO costuma deixar nos nomes das colunas
    df_results.columns = df_results.columns.str.strip()
    
    print(f"Métricas carregadas com sucesso! Total de épocas: {len(df_results)}")
    display(df_results.head(3))
else:
    print(f"Arquivo não encontrado: {RESULTS_CSV}")

## 3. Training Loss Analysis

In [ ]:
# Definindo as métricas principais de treino que queremos observar
train_losses = {
    "Box Loss": ("train/box_loss", "red"),
    "Segmentation Loss": ("train/seg_loss", "blue"),
    "Class Loss": ("train/cls_loss", "green"),
    "DFL Loss": ("train/dfl_loss", "purple")
}

fig, axes = plt.subplots(len(train_losses), 1, figsize=(10, 4 * len(train_losses)))

for i, (title, (col_name, color)) in enumerate(train_losses.items()):
    if col_name in df_results.columns:
        axes[i].plot(df_results['epoch'], df_results[col_name], color=color, linewidth=2)
        axes[i].set_title(f"Training {title}", fontsize=14)
        axes[i].set_xlabel("Epochs")
        axes[i].set_ylabel("Loss")
        axes[i].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Validation Metrics (mAP)

In [ ]:
# Gráfico das métricas AP (Average Precision) na validação
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('mAP Score', fontsize=12)

# Máscaras (Segmentation)
if 'metrics/mAP50(M)' in df_results.columns:
    ax1.plot(df_results['epoch'], df_results['metrics/mAP50(M)'], color='teal', label='Mask mAP@50', linewidth=2)
    ax1.plot(df_results['epoch'], df_results['metrics/mAP50-95(M)'], color='darkcyan', linestyle='--', label='Mask mAP@50-95', linewidth=2)

# Bounding Boxes
if 'metrics/mAP50(B)' in df_results.columns:
    ax1.plot(df_results['epoch'], df_results['metrics/mAP50(B)'], color='orange', label='Box mAP@50', linewidth=2)
    ax1.plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], color='darkorange', linestyle='--', label='Box mAP@50-95', linewidth=2)

ax1.set_title("Validation Metrics: Mask and Box mAP vs Epochs", fontsize=14)
ax1.legend(loc='lower right', fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Auto-Generated Evaluation Graphs

In [ ]:
print("=== Matriz de Confusão Normalizada ===")
display(Image(filename=LOG_DIR / "confusion_matrix_normalized.png", width=800))

print("\n=== Curva de Precision-Recall (Máscaras) ===")
display(Image(filename=LOG_DIR / "MaskPR_curve.png", width=800))

## 6. Qualitative Results (Inference on Test Set)

In [ ]:
# Carrega o melhor modelo treinado
model = YOLO(WEIGHTS_PATH)

# Pega até 5 imagens aleatórias da pasta de teste
test_images = list(TEST_IMAGES_DIR.glob("*.jpg")) # Mude para *.png se necessário
if len(test_images) > 0:
    samples = random.sample(test_images, min(5, len(test_images)))

    for img_path in samples:
        # Faz a inferência (conf=0.5 significa limite de 50% de confiança)
        results = model.predict(source=str(img_path), conf=0.5, imgsz=640, verbose=False)
        
        # O método .plot() extrai a imagem já desenhada com Bbox e Máscara!
        res_image = results[0].plot()
        res_image_rgb = cv2.cvtColor(res_image, cv2.COLOR_BGR2RGB)
        
        # Plota na tela
        plt.figure(figsize=(8, 8))
        plt.imshow(res_image_rgb)
        plt.title(f"Prediction for: {img_path.name}", fontsize=14)
        plt.axis('off')
        plt.show()
else:
    print("Nenhuma imagem encontrada na pasta de testes.")

## 7. Apresentação das Métricas Obtidas

In [ ]:
print("="*50)
print("🎯 RESULTADOS DE VALIDAÇÃO (MELHOR ÉPOCA) 🎯")
print("="*50)

if not df_results.empty:
    # Identifica a linha onde a métrica principal (Mask mAP50-95) atingiu o maior valor
    best_epoch_idx = df_results['metrics/mAP50-95(M)'].idxmax()
    best_epoch_data = df_results.iloc[best_epoch_idx]
    
    print(f"Melhor Época: {int(best_epoch_data['epoch'])}\n")
    print(f"  🟦 Mask mAP@50:    \t{best_epoch_data['metrics/mAP50(M)']:.4f}")
    print(f"  🟦 Mask mAP@50-95: \t{best_epoch_data['metrics/mAP50-95(M)']:.4f}")
    print(f"  🟩 BBox mAP@50:    \t{best_epoch_data['metrics/mAP50(B)']:.4f}")
    print(f"  🟩 BBox mAP@50-95: \t{best_epoch_data['metrics/mAP50-95(B)']:.4f}")
else:
    print("Nenhuma métrica de validação encontrada.")
print("="*50)